# 01. 小売・EC売上データセット作成
経済産業省「商業動態統計」をベースにした業態別月次販売額の時系列データを生成します。
DataRobot AutoTSでの高精度モデリングに適した特徴量設計を含みます。

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

## 設定パラメータ

In [ ]:
START_YEAR = 2015
END_YEAR = 2025
STORE_TYPES = ["百貨店", "スーパー", "コンビニ", "ドラッグストア", "EC"]
OUTPUT_PATH = Path("../data/retail_sales_dataset.csv")
np.random.seed(42)

## 外部特徴量の生成
消費者信頼感指数、CPI、気温、失業率、祝日数などマクロ経済・環境データを生成します。

In [ ]:
# 日付範囲の生成（2015-01 〜 2025-12: 132ヶ月）
date_range = pd.date_range(start=f"{START_YEAR}-01-01", end=f"{END_YEAR}-12-01", freq="MS")
n_months = len(date_range)
print(f"期間: {date_range[0].strftime('%Y-%m')} 〜 {date_range[-1].strftime('%Y-%m')} ({n_months}ヶ月)")

# 基本カラム
months = np.array([d.month for d in date_range])
years = np.array([d.year for d in date_range])

# フラグ特徴量
is_bonus_month = np.isin(months, [6, 12])
is_golden_week = (months == 5)
is_year_end = np.isin(months, [11, 12])

# 消費者信頼感指数: 2015年~42, COVID(2020)で~35に低下, その後~37-39に回復
t = np.arange(n_months)
consumer_confidence = np.full(n_months, 42.0)
for i in range(n_months):
    y = years[i]
    if y < 2020:
        # 2015-2019: 緩やかに42→40
        consumer_confidence[i] = 42.0 - (y - 2015) * 0.4
    elif y == 2020:
        # COVID急落
        consumer_confidence[i] = 35.0 + (months[i] - 1) * 0.3
    else:
        # 2021-2025: 回復 37→39
        consumer_confidence[i] = 37.0 + (y - 2021) * 0.5
consumer_confidence += np.random.normal(0, 1.0, n_months)
consumer_confidence = np.round(consumer_confidence, 1)

# CPI: 2015年~98.5, 2020年まで~99-100, その後上昇し2025年~105-108
cpi = np.full(n_months, 98.5)
for i in range(n_months):
    y = years[i]
    m = months[i]
    if y <= 2020:
        cpi[i] = 98.5 + (y - 2015) * 0.3 + (m - 1) * 0.025
    else:
        cpi[i] = 100.0 + (y - 2020) * 1.5 + (m - 1) * 0.1
cpi += np.random.normal(0, 0.3, n_months)
cpi = np.round(cpi, 1)

# 東京の月別平均気温
tokyo_monthly_temp = {
    1: 5.2, 2: 5.7, 3: 8.7, 4: 13.9, 5: 18.2, 6: 21.4,
    7: 25.0, 8: 26.4, 9: 22.8, 10: 17.5, 11: 12.1, 12: 7.6
}
avg_temperature = np.array([tokyo_monthly_temp[m] for m in months])
avg_temperature += np.random.normal(0, 1.5, n_months)
avg_temperature = np.round(avg_temperature, 1)

# 失業率: 2015年~3.3, 2019年~2.2, 2020年スパイク~3.0, 2025年~2.5
unemployment_rate = np.full(n_months, 3.3)
for i in range(n_months):
    y = years[i]
    m = months[i]
    if y < 2020:
        unemployment_rate[i] = 3.3 - (y - 2015 + (m - 1) / 12) * 0.22
    elif y == 2020:
        # スパイク
        if m <= 6:
            unemployment_rate[i] = 2.2 + (m / 6) * 0.8
        else:
            unemployment_rate[i] = 3.0 - ((m - 6) / 6) * 0.2
    else:
        unemployment_rate[i] = 2.8 - (y - 2021 + (m - 1) / 12) * 0.06
unemployment_rate += np.random.normal(0, 0.1, n_months)
unemployment_rate = np.round(np.clip(unemployment_rate, 2.0, 4.0), 1)

# 祝日数（月別の基本パターン + ランダム変動）
base_holidays = {1: 1, 2: 1, 3: 1, 4: 1, 5: 2, 6: 0, 7: 1, 8: 1, 9: 2, 10: 1, 11: 1, 12: 1}
num_holidays = np.array([base_holidays[m] for m in months])
# 一部の年でランダムに±1の変動を加える
holiday_noise = np.random.choice([-1, 0, 0, 0, 1], size=n_months)
num_holidays = np.clip(num_holidays + holiday_noise, 0, 3).astype(int)

# 外部特徴量DataFrameの作成
df_external = pd.DataFrame({
    "year_month": date_range,
    "month": months,
    "is_bonus_month": is_bonus_month,
    "is_golden_week": is_golden_week,
    "is_year_end": is_year_end,
    "consumer_confidence_index": consumer_confidence,
    "cpi": cpi,
    "avg_temperature": avg_temperature,
    "unemployment_rate": unemployment_rate,
    "num_holidays": num_holidays
})

print(f"外部特徴量テーブル: {df_external.shape}")
df_external.head()

## 業態別売上データの生成
各業態ごとに、トレンド・季節性・イベント効果・ノイズを組み合わせてリアルな月次売上を生成します。

In [ ]:
def generate_sales(date_range, store_type):
    """業態別の月次売上（億円）を生成する関数"""
    n = len(date_range)
    months = np.array([d.month for d in date_range])
    years = np.array([d.year for d in date_range])
    t = np.arange(n)  # 月インデックス（0〜131）

    if store_type == "百貨店":
        base = 0.50
        trend = base * (-0.02 / 12) * t  # -2%/年
        seasonality = np.where(months == 12, base * 0.30, 0)  # 12月ピーク+30%
        bonus = np.where(np.isin(months, [6, 12]), base * 0.15, 0)  # ボーナス月+15%
        covid = np.where(years == 2020, -base * 0.20, 0)  # COVID -20%
        noise_std = base * 0.04

    elif store_type == "スーパー":
        base = 1.20
        trend = base * (0.01 / 12) * t  # +1%/年
        seasonality = np.where(np.isin(months, [7, 8]), base * 0.08, 0)  # 夏ピーク+8%
        seasonality += np.where(months == 12, base * 0.10, 0)  # 12月ピーク+10%
        bonus = np.zeros(n)
        covid = np.where(years == 2020, base * 0.05, 0)  # COVID +5%（巣ごもり需要）
        noise_std = base * 0.03

    elif store_type == "コンビニ":
        base = 0.90
        trend = base * (0.02 / 12) * t  # +2%/年
        seasonality = np.where(np.isin(months, [7, 8]), base * 0.10, 0)  # 夏ピーク+10%（飲料）
        bonus = np.zeros(n)
        covid = np.where(years == 2020, -base * 0.08, 0)  # COVID -8%
        noise_std = base * 0.03

    elif store_type == "ドラッグストア":
        base = 0.55
        trend = base * (0.05 / 12) * t  # +5%/年
        seasonality = np.where(np.isin(months, [2, 3, 4]), base * 0.08, 0)  # 春ピーク+8%（花粉症）
        bonus = np.zeros(n)
        covid = np.where(years == 2020, base * 0.10, 0)  # COVID +10%（マスク・消毒液）
        noise_std = base * 0.03

    elif store_type == "EC":
        base = 0.15
        # 指数的成長: +15%/年
        trend = base * ((1.15 ** (t / 12)) - 1)
        seasonality = np.where(np.isin(months, [11, 12]), base * 0.20 * (1.15 ** (t / 12)), 0)  # 年末+20%
        bonus = np.zeros(n)
        # COVID 2020-2021で+40%ブースト
        covid = np.where(np.isin(years, [2020, 2021]), base * 0.40 * (1.15 ** (t / 12)), 0)
        noise_std = base * 0.05 * (1.15 ** (t / 12))  # ノイズも成長に比例

    else:
        raise ValueError(f"Unknown store type: {store_type}")

    sales = base + trend + seasonality + bonus + covid + np.random.normal(0, noise_std, n)
    sales = np.maximum(sales, 0.01)  # 負の値を防止
    return np.round(sales, 4)


# 各業態の売上を生成
all_records = []
for store_type in STORE_TYPES:
    sales = generate_sales(date_range, store_type)
    for i, d in enumerate(date_range):
        all_records.append({
            "year_month": d,
            "store_type": store_type,
            "sales_billion_yen": sales[i]
        })

df_sales = pd.DataFrame(all_records)
print(f"売上データ: {df_sales.shape}")
print(f"業態数: {df_sales['store_type'].nunique()}, 期間: {n_months}ヶ月")
print(f"合計レコード数: {len(df_sales)}")

## データの結合とプレビュー
売上データと外部特徴量をマージし、最終データセットを作成します。

In [ ]:
# 売上データと外部特徴量を結合
df = pd.merge(df_sales, df_external, on="year_month", how="left")

# ソート
df = df.sort_values(["year_month", "store_type"]).reset_index(drop=True)

print(f"最終データセット: {df.shape}")
df.head(10)

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"データセットを保存しました: {OUTPUT_PATH}")
print(f"行数: {len(df)}, カラム数: {len(df.columns)}")

In [ ]:
print("=== データセット概要 ===")
print(df.info())
print("\n=== 業態別の統計 ===")
print(df.groupby("store_type")["sales_billion_yen"].describe().round(2))
print("\n=== 先頭5行 ===")
df.head()

## DataRobot AI Catalogへのアップロード
生成したデータセットをDataRobotのAI Catalogにアップロードします。

In [ ]:
# DataRobot AI Catalogへのアップロード
try:
    from dotenv import load_dotenv
    load_dotenv("../.env")

    import datarobot as dr
    dr.Client()

    dataset = dr.Dataset.create_from_file(file_path=str(OUTPUT_PATH))
    print(f"AI Catalogにアップロード完了!")
    print(f"TRAINING_DATASET_ID={dataset.id}")
    print(f"\n.envファイルに以下を追記してください:")
    print(f"TRAINING_DATASET_ID={dataset.id}")
except Exception as e:
    print(f"DataRobotへのアップロードをスキップしました: {e}")
    print("手動でアップロードする場合は、DataRobot UIからCSVをアップロードしてください。")